In [30]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader,random_split
from PIL import Image
from torchvision.transforms import transforms

In [31]:
data_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
    )
])

In [32]:
data_dir = r'C:\Users\Lenovo\.cache\kagglehub\datasets\muhammadadeelkaggle\birds-dataset\versions\1'

In [33]:
dataset = data_dir


In [34]:
import os

In [37]:
class Parrot(Dataset):
    def __init__(self,data_dir):
        self.data = []
        self.classes = ['Amazon','Gray','Macaw','White']
        class_folder = [
            'amazon green parrot.jpg',
            'gray parrot.jpg',
            'macaw.jpg',
            'white parrot.jpg'
        ]
        data_dir = os.path.join(data_dir, 'Birds dataset.jpg')
        for label,folder in enumerate(class_folder):
            folder_path = os.path.join(data_dir, folder)
            if not  os.path.isdir(folder_path):
                print(f'Not in folder: {folder_path}')
                continue
            
            for img_Name in os.listdir(folder_path):
                if img_Name.lower().endswith(('.jpg','.png','.jpeg','.webp')):
                    self.data.append(
                        (os.path.join(folder_path,img_Name),label)
                    )
                    self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

        print("Total images loaded:", len(self.data))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, label = self.data[idx]
        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)
        return image, label
                
        
        
        
        

In [38]:
dataset = Parrot(data_dir)
dataset.classes

Total images loaded: 203


['Amazon', 'Gray', 'Macaw', 'White']

In [40]:
import torchvision.models as models
vgg16 = models.vgg16(pretrained = True)

c:\Users\Lenovo\anaconda3\envs\test\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Lenovo\anaconda3\envs\test\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to C:\Users\Lenovo/.cache\torch\hub\checkpoints\vgg16-397923af.pth


100%|██████████| 528M/528M [00:51<00:00, 10.8MB/s] 


In [41]:
dataset.classes

['Amazon', 'Gray', 'Macaw', 'White']

In [43]:
train_dataset = int(0.8*len(dataset))
test_dataset = len(dataset)-train_dataset
train_dataset ,test_dataset = random_split(dataset, [train_dataset,test_dataset])


In [44]:
train_loader = DataLoader(train_dataset,batch_size=32, pin_memory=True)
test_loader = DataLoader(test_dataset,batch_size=32,pin_memory=True)

In [66]:
epochs = 20
learning_rate = 0.0001


In [67]:
for param in vgg16.features.parameters():
    param.requires_grad = False

In [68]:
vgg16.classifier = nn.Sequential(
    nn.Linear(25088, 1024),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(1024, 512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512, 10)
)

In [69]:
import torch.optim as optim

In [70]:
optimizer = optim.Adam(vgg16.classifier.parameters(), lr=learning_rate)

In [71]:
# ensure criterion is defined (if the cell that defines it wasn't run)
if 'criterion' not in globals():
    criterion = nn.CrossEntropyLoss()

vgg16.train()
for epoch in range(epochs):
    total_loss = 0.0
    for images, label in train_loader:
        outputs = vgg16(images)
        print(outputs.shape)
        print(label.shape)
        loss = criterion(outputs, label)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        break
    avg_loss = total_loss / len(train_loader)
    print(f'Epoch: {epoch+1}/{epochs} avg_loss: {avg_loss}')
        

torch.Size([32, 10])
torch.Size([32])
Epoch: 1/20 avg_loss: 0.3742319345474243
torch.Size([32, 10])
torch.Size([32])
Epoch: 2/20 avg_loss: 0.2956877152125041
torch.Size([32, 10])
torch.Size([32])
Epoch: 3/20 avg_loss: 0.21294444799423218
torch.Size([32, 10])
torch.Size([32])
Epoch: 4/20 avg_loss: 0.16023059686024985
torch.Size([32, 10])
torch.Size([32])
Epoch: 5/20 avg_loss: 0.11675951878229777
torch.Size([32, 10])
torch.Size([32])
Epoch: 6/20 avg_loss: 0.08411481976509094
torch.Size([32, 10])
torch.Size([32])
Epoch: 7/20 avg_loss: 0.06386749943097432
torch.Size([32, 10])
torch.Size([32])
Epoch: 8/20 avg_loss: 0.04468036691347758
torch.Size([32, 10])
torch.Size([32])
Epoch: 9/20 avg_loss: 0.029230445623397827
torch.Size([32, 10])
torch.Size([32])
Epoch: 10/20 avg_loss: 0.020906853179136913
torch.Size([32, 10])
torch.Size([32])
Epoch: 11/20 avg_loss: 0.021740165849526722
torch.Size([32, 10])
torch.Size([32])
Epoch: 12/20 avg_loss: 0.011504555741945902
torch.Size([32, 10])
torch.Size([32

In [72]:
# Accuracy

vgg16.eval()
total = 0
corrected = 0
for images , labels in test_loader:
    outputs = vgg16(images)
    _,predicted = torch.max(outputs,dim = 1)
    total += labels.size(0)
    corrected += (predicted == labels).sum().item()
accuracy = corrected/total

print(f'Test Accuracy : {accuracy*100:.2f}%')

Test Accuracy : 78.05%
